In [2]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

df = pd.read_csv('data/spectral_feature_data.csv')
target_cols = [col for col in df.columns if col.startswith("p")]

# Select features: All columns that are NOT in the exclusion list
# This assumes your CSV contains: Spectral_Cols, ph, ec, Target_Cols, and ID
non_feature_cols = [col for col in df.columns if col.startswith('p4')]

feature_cols = [col for col in df.columns if col not in non_feature_cols]

#print(f"{target_cols}\n\n{non_feature_cols}\n\n{feature_cols}")
 
#input features
spectral_columns = [col for col in feature_cols if not col.startswith("p")]

In [5]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

results = []

# Assuming target_cols and spectral_columns are already defined
# Example: spectral_columns = ['410', '435', '460', '485', ...]

combinations = {"Spectral Only": [False, False],
                "Spectral + pH": [True, False],
                "Spectral + EC": [False, True],
                "Spectral + pH + EC": [True, True]}

imputer = SimpleImputer(strategy='mean')

# Define Optuna objective function for Random Forest
def objective(trial, X_tr, X_te, y_tr, y_te):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 1.0])
    }
    
    # n_jobs=-1 speeds up training by using all CPU cores
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    
    return mean_squared_error(y_te, preds)

# Suppress Optuna's default print statements to keep terminal output clean
optuna.logging.set_verbosity(optuna.logging.WARNING)

for config_type in combinations.keys():
    print(f"\n========== Evaluating Config: {config_type} ==========")
    
    prediction_columns = spectral_columns.copy()
    
    maskpH = pd.Series(True, index=df.index)
    maskEC = pd.Series(True, index=df.index)

    if combinations[config_type][0]: # if we are to use ph value
        prediction_columns.append("p1.pH.index")
        maskpH = df["p1.pH.index"].notna()  
        
    if combinations[config_type][1]: # if we are to use EC value
        prediction_columns.append("p1.EC.ds_m")
        maskEC = df["p1.EC.ds_m"].notna()   

    # Update X for this specific configuration
    X = df[prediction_columns]
    
    # Combine the feature masks (rows where required features exist)
    feature_mask = maskpH & maskEC
    
    print(f"Features in use: {len(prediction_columns)} columns")
    
    for target in target_cols:
        
        if target not in df.columns:
            continue
            
        # Combine target mask with the feature masks properly
        target_mask = df[target].notna()
        final_mask = target_mask & feature_mask 
        
        y_clean = df.loc[final_mask, target]
        X_clean = X.loc[final_mask]
        
        if y_clean.shape[0] < 100:
            print(f"  -> Skipping {target}: Not enough data ({y_clean.shape[0]} rows).")
            continue
            
        # Impute NaNs and retain DataFrame structure to prevent feature name warnings
        X_clean_imputed = pd.DataFrame(imputer.fit_transform(X_clean), columns=X_clean.columns)

        X_train, X_test, y_train, y_test = train_test_split(X_clean_imputed, y_clean, test_size=0.2, random_state=42)

        # Run Optuna study for this target and configuration
        study = optuna.create_study(direction='minimize')
        
        try:
            # 30 trials is a good baseline for RF, adjust up if needed
            study.optimize(lambda trial: objective(trial, X_train, X_test, y_train, y_test), n_trials=30) 
        except Exception as e:
            print(f"  -> Error training {target}: {e}")
            continue
        
        best_params = study.best_params
        
        # Train final model with the optimized parameters
        best_rf = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)
        best_rf.fit(X_train, y_train)
        
        y_pred = best_rf.predict(X_test)
        
        metrics = {
            'Config': config_type, 
            'Feature': target,
            'Best_Params': str(best_params), 
            'R2': r2_score(y_test, y_pred),
            'MAE': mean_absolute_error(y_test, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            "MPE": np.mean((y_test - y_pred) / y_test) * 100
        }
        
        results.append(metrics)
        print(f"  -> Finished {target}: R2 = {metrics['R2']:.3f} | MAE = {metrics['MAE']:.3f}")

# Save and View Performance Table
if results:
    performance_df = pd.DataFrame(results)
    performance_df.to_csv('rf_optuna_performance_results_combinations.csv', index=False)
    print("\n--- Random Forest Performance Summary ---")
    print(performance_df)


========== Evaluating Config: Spectral Only ==========
Features in use: 18 columns
  -> Finished p1.pH.index: R2 = 0.543 | MAE = 0.749
  -> Finished p1.EC.ds_m: R2 = 0.173 | MAE = 0.166
  -> Finished p1.Clay.wt_pct: R2 = 0.334 | MAE = 14.112
  -> Finished p1.Sand.wt_pct: R2 = 0.279 | MAE = 7.667
  -> Finished p1.Silt.wt_pct: R2 = 0.255 | MAE = 14.319
  -> Finished p2.N.wt_pct: R2 = 0.688 | MAE = 0.123
  -> Skipping p2.Zn.mg_kg: Not enough data (0 rows).
  -> Finished p2.OC.wt_pct: R2 = 0.751 | MAE = 1.957
  -> Skipping p3.Fe.mg_kg: Not enough data (0 rows).
  -> Finished p3.K.mg_kg: R2 = 0.114 | MAE = 224.931
  -> Finished p3.P.mg_kg: R2 = 0.088 | MAE = 20.988
  -> Finished p3.S.wt_pct: R2 = 0.073 | MAE = 0.012
  -> Finished p4.BD.g_cm3: R2 = 0.311 | MAE = 0.184
  -> Finished p4.CEC.cmolc_kg: R2 = 0.269 | MAE = 7.841
  -> Finished p4.CF.wt_pct: R2 = 0.061 | MAE = 8.899
  -> Finished p4.WR_10kPa.wt_pct: R2 = 0.180 | MAE = 8.303
  -> Finished p4.WR_1500kPa.wt_pct: R2 = 0.174 | MAE = 7.6